In [2]:
#!pip install -U transformers

In [19]:
#!pip install medmnist

In [23]:
!pip install -q bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00


## Local Inference on GPU
Model page: https://huggingface.co/google/medgemma-4b-it

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/medgemma-4b-it)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [24]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

# 4-bit quantization — fits on free T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="cuda",       # force ALL layers to GPU
    quantization_config=bnb_config
)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [25]:
from medmnist import PneumoniaMNIST
from PIL import Image
import numpy as np

dataset = PneumoniaMNIST(split='test', download=True)

def preprocess_image(img):
    # img is already a PIL Image object from dataset[0]
    img = img.convert("RGB")
    img = img.resize((224, 224))
    return img

image, label = dataset[1]
image = preprocess_image(image)

In [26]:
prompt = """
You are an experienced radiologist.

Analyze the chest X-ray image carefully.

Provide:
1. Radiological findings
2. Presence or absence of pneumonia
3. Severity level (Mild/Moderate/Severe)
4. Clinical impression

Be concise and medically accurate.
"""

In [27]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": """You are an experienced radiologist.

Analyze this chest X-ray image.

Provide:
1. Radiological findings
2. Presence or absence of pneumonia
3. Severity
4. Clinical impression.

Be concise and medically accurate."""}
        ]
    }
]

In [28]:
prompt = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False
)


In [29]:
inputs = processor(
    text=prompt,
    images=image,
    return_tensors="pt"
).to("cuda")


In [30]:
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        min_new_tokens=50,
        do_sample=False,
        repetition_penalty=1.1
    )



In [20]:
print(type(image), image.size, image.mode)  # Should be: PIL, (224,224), RGB

<class 'PIL.Image.Image'> (224, 224) RGB


In [31]:
generated_tokens = output[0, inputs["input_ids"].shape[1]:]
response = processor.decode(generated_tokens, skip_special_tokens=True)
print(response)

Here's my analysis of the chest X-ray provided:

**1. Radiological Findings:**

*   The image shows a clear view of the lung fields, though with some artifacts present. The heart size appears within normal limits based on the available information. There is no obvious evidence of significant consolidation, pleural effusion, or pneumothorax.

**2. Absence of Pneumonia:**

*   No definitive signs of pneumonia (e.g., consolidation, infiltrates) are visible in the lungs.

**3. Severity:**

*   I cannot assess severity without more clinical context. However, given the lack of any apparent abnormalities, I would consider it to be *not severe*.

**4. Clinical Impression:**

*   Based solely on this single image, there is no indication of acute cardiopulmonary disease. Further evaluation may be warranted depending on the patient's symptoms and history.

